# C-SHAP walkthrough with the `cshap` package

This notebook reproduces the core of `01_original_c_shap_vs_shap.ipynb` with the reusable package in `src/cshap`, and adds the one thing the original comparison could not show: how far each explanation is from the **exact** SHAP values computed with the full training background.

* **Standard SHAP**: Kernel SHAP with 100 random training rows as background (`shap.sample`).
* **C-SHAP** (as defined in this repository): Kernel SHAP with the 20 K-means centroids of the standardized training data as background.
* **Exact reference**: interventional Tree SHAP with all 455 training rows, which is the quantity both Kernel SHAP runs approximate.

Run `pip install -e .` from the repository root first.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from cshap import CShapExplainer, exact_shap, full_background, kernel_shap, random_background
from cshap.data import prepare_data
from cshap.metrics import global_metrics, instance_metrics
from cshap.models import fit_model, output_function

data = prepare_data("breast_cancer", max_reference=None)       # same split and scaling as the notebook
model = fit_model("rf", data.X_train, data.y_train, seed=42)   # RandomForest(200 trees, seed 42)
f = output_function(model, "rf")                              # P(benign)
X_explain = data.X_test[:20]
print("test accuracy:", (model.predict(data.X_test) == data.y_test).mean())

test accuracy: 0.956140350877193


## Explain with C-SHAP and with the standard background

In [2]:
cshap = CShapExplainer(f, n_clusters=20, random_state=42).fit(data.X_train)
res_cshap = cshap.explain(X_explain, seed=0)

res_std = kernel_shap(f, random_background(data.X_train, 100, seed=42), X_explain, seed=0)

print(f"C-SHAP:   {res_cshap.seconds:5.2f} s Kernel SHAP + {cshap.background_.fit_seconds:.2f} s K-means")
print(f"Standard: {res_std.seconds:5.2f} s Kernel SHAP")

# efficiency: attributions add up to the prediction minus the base value
print("max efficiency gap:", np.abs(res_cshap.values.sum(1) + res_cshap.base_value - f(X_explain)).max())

C-SHAP:    3.95 s Kernel SHAP + 0.04 s K-means
Standard: 19.38 s Kernel SHAP
max efficiency gap: 2.7755575615628914e-17


## Compare both against the exact full-background SHAP values

In [3]:
exact, exact_base = exact_shap(model, "rf", X_explain, full_background(data.X_train))

rows = []
for name, res in [("Standard SHAP (100 random rows)", res_std), ("C-SHAP (20 centroids)", res_cshap)]:
    inst = instance_metrics(res.values, exact)
    glob = global_metrics(res.values, exact)
    rows.append({"method": name, "seconds": res.seconds,
                 "per-instance rel. L1 error": inst["rel_l1"].mean(),
                 "per-instance Spearman": np.nanmean(inst["spearman"]),
                 "global Pearson": glob["pearson"], "global top-10 overlap": glob["top10"]})
pd.DataFrame(rows).round(3)

,method,seconds,per-instance rel. L1 error,per-instance Spearman,global Pearson,global top-10 overlap
0,Standard SHAP (100 random rows),19.376,0.07,0.972,1.000,0.9
1,C-SHAP (20 centroids),3.950,0.31,0.921,0.998,0.9


Global importance correlates highly with the exact values for both backgrounds (the statistic the original notebook reported). Per-instance errors separate the methods much more clearly. The full multi-dataset analysis, with seeds, significance tests and size-matched random controls, is produced by `cshap run all` and summarised in the README.

In [4]:
top = np.argsort(-np.abs(exact).mean(0))[:8]
pd.DataFrame({
    "feature": np.array(data.feature_names)[top],
    "exact": np.abs(exact).mean(0)[top],
    "standard": np.abs(res_std.values).mean(0)[top],
    "c-shap": np.abs(res_cshap.values).mean(0)[top],
}).round(4)

,feature,exact,standard,c-shap
0,worst area,0.0663,0.0663,0.0680
1,worst perimeter,0.0639,0.0643,0.0643
2,worst radius,0.0532,0.0532,0.0560
3,worst concave points,0.0511,0.0516,0.0536
4,mean concave points,0.0323,0.0327,0.0315
5,mean concavity,0.0258,0.0257,0.0250
6,worst concavity,0.0256,0.0247,0.0237
7,area error,0.0224,0.0229,0.0213
